# Baseline

本ノートブックでは、最小限の特徴量と LightGBM を用いてベースラインモデルを作成する。
データ読み込みから学習・CV・提出までの一連の流れをまず組み、以降の改善の基準となるスコアを得ることが目的である。

ベースラインは `src/` に依存せず、このノートブック単体で実行・提出まで完結する構成としている。

In [13]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import lightgbm as lgb
import warnings
warnings.filterwarnings("ignore")
%matplotlib inline

SEED = 123

## 1. データ読み込み

In [14]:
train_df = pd.read_csv("../data/train.csv")
print("shape:", train_df.shape)
train_df.head()

shape: (1460, 81)


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


## 2. 特徴量定義

ベースラインでは、数値型で欠損値がなく、追加の前処理なしでそのまま使える変数を 2 つ選んだ。

- `OverallQual`：住宅の全体品質（1〜10）
- `MSSubClass`：建物タイプの分類コード（数値型だが実質カテゴリ変数。ベースラインでは前処理なしでそのまま使用する）

目的変数 `SalePrice` には `log1p` 変換を適用する。
本コンペの評価指標が予測値と実測値の対数に対する RMSE であるため、学習時点で対数変換しておく。

In [15]:
features = ["OverallQual", "MSSubClass"]
X_train = train_df[features]
y_train = np.log1p(train_df["SalePrice"])

In [16]:
print(X_train.shape)
print(y_train.shape)

(1460, 2)
(1460,)


## 3. モデル学習・CV

LightGBM の 5-Fold CV でベースラインの精度を確認する。
early stopping を 100 ラウンドに設定し、過学習を抑えながら適切なイテレーション数を自動決定する。

In [17]:
params = {
    "boosting_type" : "gbdt",
    "objective" : "regression",
    "metric" : "rmse",
    "learning_rate" : 0.1,
    "num_leaves" : 16,
    "n_estimators" : 1000,
    "random_state" : SEED,
    "importance_type" : "gain",
}

In [18]:
n_splits = 5
cv = KFold(n_splits=n_splits, shuffle=True, random_state=SEED)

metrics = []
imp = pd.DataFrame()

for nfold, (train_idx, val_idx) in enumerate(cv.split(X_train)):
  x_tr, y_tr = X_train.iloc[train_idx], y_train.iloc[train_idx]
  x_va, y_va = X_train.iloc[val_idx], y_train.iloc[val_idx]

  print("学習用:", x_tr.shape, y_tr.shape)
  print("評価用:", x_va.shape, y_va.shape)

  model = lgb.LGBMRegressor(**params)
  model.fit(
    x_tr,
    y_tr,
    eval_set=[(x_tr, y_tr), (x_va,y_va)],
    callbacks=[
        lgb.early_stopping(stopping_rounds=100, verbose=True),
        lgb.log_evaluation(0),
    ])

  y_tr_pred = model.predict(x_tr)
  y_va_pred = model.predict(x_va)

  rmse_tr = np.sqrt(mean_squared_error(y_tr, y_tr_pred))
  rmse_va = np.sqrt(mean_squared_error(y_va, y_va_pred))

  print("[RMSE] tr: {:.5f}, va: {:.5f}".format(rmse_tr, rmse_va))

  metrics.append([nfold, rmse_tr, rmse_va])

  _imp = pd.DataFrame({
        "col": X_train.columns,
        "imp": model.feature_importances_,
        'nfold': nfold
        })

  imp = pd.concat([imp, _imp], axis=0, ignore_index=True)

学習用: (1168, 2) (1168,)
評価用: (292, 2) (292,)
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000320 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 26
[LightGBM] [Info] Number of data points in the train set: 1168, number of used features: 2
[LightGBM] [Info] Start training from score 12.021806
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[82]	training's rmse: 0.205543	valid_1's rmse: 0.202439
[RMSE] tr: 0.20554, va: 0.20244
学習用: (1168, 2) (1168,)
評価用: (292, 2) (292,)
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000094 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 25
[LightGBM] [Info] Number of data points in the train set: 1168, number of used features: 2
[LightGBM] [Info] Start training from score 12.014343
Training until validation scores don't improve fo

In [19]:
metrics = np.array(metrics)
print("[cv] tr: {:.5f}+-{:.5f}, va: {:.5f}+-{:.5f}".format(
    metrics[:,1].mean(), metrics[:,1].std(),
    metrics[:,2].mean(), metrics[:,2].std(),
))

[cv] tr: 0.20560+-0.00318, va: 0.21155+-0.00831


## 4. 特徴量重要度

In [20]:
# 特徴量重要度
imp = imp.groupby("col")["imp"].agg(["mean", "std"])
imp.columns = ["imp", "imp_std"]
imp_df = imp.sort_values(by="imp", ascending=False)
imp_df

,imp,imp_std
col,,
OverallQual,649.948566,14.607817
MSSubClass,70.142910,5.246156


## 5. テストデータの予測・提出

In [21]:
# 全データで再学習して提出
model = lgb.LGBMRegressor(**params)
model.fit(X_train, y_train)

test_df = pd.read_csv("../data/test.csv")
X_test = test_df[features]

y_test_pred = np.expm1(model.predict(X_test))

submission = pd.DataFrame({
    "Id": test_df["Id"],
    "SalePrice": y_test_pred,
})
submission.to_csv("../submissions/submission_00_baseline.csv", index=False)
print("Saved!")

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000121 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 26
[LightGBM] [Info] Number of data points in the train set: 1460, number of used features: 2
[LightGBM] [Info] Start training from score 12.024057
Saved!


## 6. Summary

本ノートブックでは、2 変数（`OverallQual`、`MSSubClass`）と LightGBM によるベースラインモデルを作成した。

| 指標 | スコア |
|---|---:|
| CV-RMSE | 0.211 |
| LB | 0.218 |

このスコアを起点に、以降のノートブックで前処理・特徴量エンジニアリング・モデリング・アンサンブルを段階的に行い、精度を改善していく。